# Période et stabilité hors échantillon

Ce notebook reprend les deux modes et le graphique de `folded_profile.ipynb`.
Il charge **tout le CSV**, choisit la période sur une première portion chronologique,
puis compare cette période à celles estimées séparément dans les blocs de test.

- `events` : colonne `timestamp`, une ligne par événement.
- `values` : colonnes `timestamp,value`, une ligne par mesure.

Le CSV est déjà nettoyé et trié : dates UTC ISO 8601 avec suffixe `Z`, valeurs numériques
finies, sans données manquantes. Les observations simultanées sont conservées.
Dépendances : pandas, NumPy, matplotlib.

## 1. Paramètres

`train_fraction=0.25` réserve le premier quart de la **durée**, pas des lignes, à la
recherche. `test_blocks=3` découpe le reste en trois intervalles de même durée pour
éviter qu'une moyenne globale masque une dégradation au cours du temps.
Choisir ces réglages avant d'examiner les résultats.


In [ ]:
from math import ceil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

csv_path = 'timeseries.csv'
mode = 'values'
p_min, p_max = (1.0, 40.0)
phase_bins = 100
train_fraction = .5
test_blocks = 3
value_label = 'Lifetime (ms)'
title = Path(csv_path).stem
plt.rcParams['figure.dpi'] = 120
cyan, gold, dark = ('#62e4df', '#f3cf86', '#080b17')

## 2. Lecture et séparation chronologique

La frontière est calculée entre le premier et le dernier timestamp. Les lignes avant
la frontière constituent l'apprentissage ; celles à la frontière ou après constituent
le test. Un groupe de timestamps identiques reste donc dans une seule partie.

En mode événements, l'intervalle entier est supposé observé sans interruption :
les absences d'événements valent zéro. Les périodes silencieuses hors du CSV sont inconnues.
La même origine temporelle est utilisée dans les deux parties.


In [ ]:
data = pd.read_csv(csv_path)
dates = pd.to_datetime(data.timestamp, format = 'ISO8601', utc = True)
timestamps = pd.DatetimeIndex(dates).as_unit('ns').asi8
elapsed = timestamps-timestamps[0]
times = elapsed/1e9
values = data.value.to_numpy(float) if mode == 'values' else None
def split_series(times, fraction):
  """split elapsed time without separating simultaneous observations"""
  if not 0 < fraction < 1: raise ValueError('Use a fraction between 0 and 1.')
  end = np.nextafter(times[-1], np.inf)
  boundary = times[-1]*fraction
  train = times < boundary
  if train.sum() < 3 or (~train).sum() < 3:
    raise ValueError('Both parts need at least three observations.')
  return boundary, end, train
split, end, train = split_series(times, train_fraction)
train_values = values[train] if values is not None else None
print(f"Learning: {train.sum():,} rows | Test: {(~train).sum():,} rows")

## 3. Recherche sur la première portion uniquement

Même balayage régulier en fréquence et mêmes scores que dans le notebook original :
**χ² de Pearson** pour les événements, **L de Davies** pour les valeurs.
Le pas de grille dépend uniquement de la durée d'apprentissage. Au moins cinq cycles
doivent tenir dans cette portion. Le calcul se fait par lots.

Les instants et valeurs de la seconde partie ne servent pas à sélectionner la période.
Pour les événements, l'exposition d'apprentissage s'arrête exactement à la frontière,
et non au dernier événement observé avant celle-ci.

Références : [Leahy et al., 1983](https://adsabs.harvard.edu/pdf/1983ApJ...272..256L)
et [Davies, 1990](https://adsabs.harvard.edu/pdf/1990MNRAS.244...93D).


In [48]:
def phase_labels(times, periods, bins):
  """assign each observation to a phase bin"""
  phase = np.remainder(times, periods)/periods
  labels = (phase*bins).astype(int)
  
  return np.minimum(labels, bins-1)

def phase_exposure(periods, span, bins):
  """compute exposure for each phase over the full file"""
  periods = np.asarray(periods).reshape(-1, 1)
  cycles = np.floor(span/periods)
  remaining = span-cycles*periods
  phase_start = np.arange(bins)*(periods/bins)
  partial_exposure = np.clip(remaining-phase_start, 0, (periods/bins))
  
  return cycles*(periods/bins) + partial_exposure

def period_scores(times, values, periods, bins, span):
  """compute pearson or davies scores for a batch of periods"""
  labels = phase_labels(times, periods[:, None], bins)
  offsets = np.arange(len(periods))[:, None]*bins
  groups = labels + offsets
  
  flat_groups = groups.ravel()
  counts = np.bincount(flat_groups, minlength = len(periods)*bins)
  counts = counts.reshape(len(periods), bins)
  
  if values is None:
    exposure = phase_exposure(periods, span, bins)
    expected = (len(times)/span)*exposure
    deviation = counts - expected
    contributions = deviation**2/expected
    return contributions.sum(axis = 1)
  
  centered = values-values.mean()
  repeated = np.tile(centered, len(periods))
  totals = np.bincount(flat_groups, weights = repeated,
    minlength = len(periods)*bins).reshape(len(periods), bins)
  
  means = totals/np.maximum(counts, 1)
  fitted = means.ravel()[groups]
  
  residual = centered-fitted
  
  between = np.sum(counts*means**2, axis = 1)
  within = np.sum(residual**2, axis = 1)
  occupied = (counts > 0).sum(axis = 1)
  between_df = occupied-1
  within_df = len(times)-occupied
  
  with np.errstate(divide = 'ignore', invalid = 'ignore'):
    between_variance = between/between_df
    within_variance = within/within_df
    scores = between_variance/within_variance
  invalid = (between_df <= 0) | (within_df <= 0)
  scores[invalid] = np.nan
  
  return scores

def find_period(times, values, p_min, p_max, bins, span):
  """select the best period on a uniform frequency grid"""
  p_max = min(p_max, span/5)
  if not 0 < p_min <= p_max or bins < 2:
    raise ValueError('Invalid search range for the learning interval.')

  trials = ceil((1/p_min-1/p_max)*4*bins*span)+1
  frequencies = np.linspace(1/p_min, 1/p_max, trials)
  periods = 1/frequencies
  scores = np.full(trials, np.nan)
  batch = max(1, 262_144//len(times))
  
  for start in range(0, trials, batch):
    stop = min(start+batch, trials)
    candidates = periods[start:stop]
    scores[start:stop] = period_scores(times, values, candidates, bins, span)
    
  if np.isnan(scores).all(): raise ValueError('No identifiable period.')
  best = np.nanargmax(scores)
  return float(periods[best])

period_s = find_period(times[train], train_values, p_min, p_max,
  phase_bins, split)
print(f"Frozen period: {period_s:.9f} s")

Frozen period: 29.953807137 s


## 4. Comparer les périodes
Chaque bloc de test fait l'objet d'une recherche indépendante, avec les mêmes
scores, bins et bornes que l'apprentissage, limités à cinq cycles par bloc.
L'origine des temps est ramenée au début du bloc pour calculer son exposition.
Le profil, le niveau moyen et l'amplitude peuvent changer.
`period_tolerance = .01` accepte un écart de **1 %** à la période d'apprentissage.
Le tableau donne la période du bloc, son écart signé en secondes et en pourcentage,
puis le résultat de cette comparaison. Un bloc trop court, trop peu fourni ou sans
score défini est marqué `indeterminate`.
Cette tolérance est un seuil pratique, pas une incertitude statistique ni une
preuve de périodicité. Les blocs courts ou bruités peuvent sélectionner un alias
ou une harmonique. L'accord des meilleures périodes ne teste pas la stabilité
de phase et n'impose pas que les profils soient identiques.
Les estimations de test ne remplacent pas la période d'apprentissage dans le
graphique final.


In [ ]:
period_tolerance = .01
def test_stability(times, values, period, bins, split, end, blocks,
  p_min, p_max, tolerance):
  """compare independently estimated periods across test blocks"""
  edges = np.linspace(split, end, blocks+1)
  rows = []
  for index, (start, stop) in enumerate(zip(edges[:-1], edges[1:]), 1):
    mask = (times >= start) & (times < stop)
    size = int(mask.sum())
    estimate, change = np.nan, np.nan
    status = 'indeterminate'
    block_values = values[mask] if values is not None else None
    estimate = find_period(times[mask]-start, block_values,
      p_min, p_max, bins, stop-start)
    change = (estimate-period)/period
    status = 'within tolerance' if abs(change) <= tolerance else 'different'
    reason = ''
    rows.append({'block': index, 'start_s': start, 'end_s': stop,
      'rows': size, 'period_s': estimate, 'difference_s': estimate-period,
      'difference_pct': 100*change, 'status': status, 'reason': reason})
  
  return pd.DataFrame(rows)

stability = test_stability(times, values, period_s, phase_bins,
  split, end, test_blocks, p_min, p_max, period_tolerance)
display(stability)
print(f"Period tolerance: {100*period_tolerance:g}%")
print(stability.status.value_counts().to_string())


,block,start_s,end_s,rows,period_s,difference_s,difference_pct,status,reason
0,1,1754.971879,2339.962505,109,3.337054,-26.648092,-88.870977,different,
1,2,2339.962505,2924.953131,89,32.467213,2.482068,8.277658,different,
2,3,2924.953131,3509.943757,88,15.100771,-14.884375,-49.639162,different,


Period tolerance: 1%
status
different    3


## 5. Replier l'intégralité des données
Le graphique utilise **toutes les observations**, à la période sélectionnée sur
l'apprentissage. La courbe supérieure est la moyenne de toute la série.
Les périodes estimées dans les blocs de test ne modifient pas ce graphique.
Les événements donnent un taux par seconde d'exposition ; les valeurs donnent une moyenne.
Les cases sans observation restent manquantes, sans interpolation.


In [ ]:
def fold_series(times, values, period, bins):
  """build the profile and cycle matrix with their weights"""
  span = np.nextafter(times[-1], np.inf)
  cycles = np.floor(times/period).astype(int)
  phases = phase_labels(times, period, bins)
  rows = cycles.max()+1
  cell_count = rows*bins
  
  if cell_count > 2_000_000: raise MemoryError('Reduce the heatmap size.')
  cells = cycles*bins+phases
  counts = np.bincount(cells, minlength = cell_count).reshape(rows, bins)
  
  if values is None:
    totals = counts
    cycle_start = np.arange(rows)[:, None]*period
    phase_start = np.arange(bins)*period/bins
    weights = np.clip(span-cycle_start-phase_start, 0, period/bins)
  else:
    totals = np.bincount(cells, weights = values, minlength = cell_count)
    totals = totals.reshape(rows, bins)
    weights = counts
  
  matrix = totals/np.where(weights > 0, weights, np.nan)
  phase_totals = totals.sum(axis = 0)
  phase_weights = weights.sum(axis = 0)
  profile = phase_totals/np.where(phase_weights > 0, phase_weights, np.nan)
  
  return profile, matrix, weights

profile, matrix, weights = fold_series(times, values, period_s, phase_bins)
feature = 'Events / second' if mode == 'events' else value_label

## 6. Graphique et frontière apprentissage/test

Même profil cyan, heatmap magma et échelle logarithmique que dans `folded_profile.ipynb`.
La ligne pointillée marque la frontière au cycle `split / period_s` :
apprentissage au-dessus, test en dessous.

Les cycles contigus sont regroupés pour l'affichage, avec leurs poids.
Le cycle qui contient la frontière reste isolé du regroupement ; s'il est partiel,
il contient des observations des deux parties. La ligne donne la position de la
frontière en coordonnées de cycle.


In [ ]:
rows, bins = matrix.shape
edges = np.linspace(0, rows, min(250, rows)+1, dtype = int)
split_cycle = split/period_s
border = [int(np.floor(split_cycle)), int(np.ceil(split_cycle))]
edges = np.unique(np.r_[edges, np.clip(border, 0, rows)])
totals = np.nan_to_num(matrix)*weights
totals = np.add.reduceat(totals, edges[:-1], axis = 0)
counts = np.add.reduceat(weights, edges[:-1], axis = 0)
means = totals/np.where(counts > 0, counts, np.nan)

colors = np.sign(means)*np.log1p(np.abs(means))
phases = np.linspace(0, 1, bins+1)
centers = (phases[:-1]+phases[1:])/2
cmap = plt.get_cmap('magma').with_extremes(bad = '#303342')

with plt.style.context('dark_background'):
  fig, (top, bottom) = plt.subplots(2, 1, figsize = (13, 7),
    sharex = True, height_ratios = [1, 3], constrained_layout = True)
  top.plot(centers, profile, color = cyan)
  top.set(ylabel = feature, title = f"{title} | P = {period_s:.6f} s")

  mesh = bottom.pcolormesh(phases, edges, colors, cmap = cmap)
  bottom.axhline(split_cycle, color = gold, linestyle = '--',
    label = 'Learning / test')
  bottom.legend(loc = 'upper right')
  bottom.invert_yaxis()
  bottom.set(xlabel = 'Phase (cycles)', ylabel = 'Cycle number', xlim = (0, 1))
  scale = 'signed log(1+|mean|)' if np.any(means < 0) else 'log(1+mean)'
  fig.colorbar(mesh, ax = [top, bottom], label = f"{scale}: {feature}")
  plt.show()